In [23]:
import os
import re
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from collections import Counter

# ⚠️ 중요: 반드시 konlpy/tag 임포트 '전'에 실행해야 적용됩니다.
# JVM의 최대 힙 메모리를 4기가바이트(-Xmx4g)로 확장 설정
os.environ['JVM_ARGS'] = '-Xmx4g'
import konlpy
from konlpy.tag import Mecab, Kkma, Okt
from gensim.models.word2vec import Word2Vec
import sentencepiece as spm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.utils.rnn import pad_sequence


# init config
# 한글 폰트 설정 (주피터 노트북 시각화 대응)
# !sudo apt-get install -y fonts-nanum
# !sudo fc-cache -fv
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

# 디바이스 지정 (GPU 가속 지원)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

# 데이터셋 기본 경로
base_path = os.path.join(os.getcwd(), '..', '..', 'work', 'sentiment_classification')


print(torch.__version__)
print(np.__version__)
print(konlpy.__version__)

device: cuda
2.7.1+cu126
1.26.4
0.6.0


In [ ]:
# define

# ---------------------------------------------------------------------------
# [정밀 매핑] 제공된 sp_tokenize 기반 및 가변 경로 제어 보완
# ---------------------------------------------------------------------------
def sp_tokenize(s, corpus, vocab_dir):
    """
    s: 학습 및 로드 완료된 sentencepiece 모델 인스턴스
    corpus: 문장 데이터 리스트 (pd.Series 혹은 list)
    vocab_dir: .vocab 파일이 저장된 절대/상대 디렉토리 경로
    """
    tensor = []
    for sen in corpus:
        if not isinstance(sen, str):
            sen = str(sen) if pd.notnull(sen) else ""
        tensor.append(s.EncodeAsIds(sen))

    # 지정된 경로에서 .vocab 파일 확보
    vocab_path = os.path.join(vocab_dir, "korean_spm.vocab")
    if not os.path.exists(vocab_path):
        # 다중 튜닝 실험 시 가변 파일명이 생성되므로 폴더 내 매칭 유연화
        import glob
        found_vocabs = glob.glob(os.path.join(vocab_dir, "*.vocab"))
        if found_vocabs:
            vocab_path = found_vocabs[0]
        else:
            raise FileNotFoundError(f"⚠️ {vocab_dir} 디렉토리 내에 .vocab 파일이 존재하지 않습니다.")

    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = f.readlines()

    word_index = {}
    index_word = {}

    for idx, line in enumerate(vocab):
        word = line.split("\t")[0]
        word_index[word] = idx
        index_word[idx] = word

    # 빈 리스트 변환 및 다차원 정수 스태킹 처리
    tensor_tensors = [torch.tensor(t) if len(t) > 0 else torch.tensor([0]) for t in tensor]
    tensor = pad_sequence(tensor_tensors, batch_first=True, padding_value=0)

    return tensor, word_index, index_word


def train_spm_tokenizer(input_txt_path, vocab_size, model_type, output_dir):
    """지정된 디렉토리에 SentencePiece 모델을 학습하여 저장하고 인스턴스를 반환합니다."""
    os.makedirs(output_dir, exist_ok=True)
    model_prefix = os.path.join(output_dir, f"spm_{model_type}_{vocab_size}")
    
    templates = (
        f"--input={input_txt_path} "
        f"--model_prefix={model_prefix} "
        f"--vocab_size={vocab_size} "
        f"--model_type={model_type} "
        f"--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3"
    )
    spm.SentencePieceTrainer.Train(templates)
    
    sp_instance = spm.SentencePieceProcessor()
    sp_instance.Load(f"{model_prefix}.model")
    return sp_instance


# ---------------------------------------------------------------------------
# DataHandler: KoNLPy 3종 분기 및 SentencePiece 인터페이스 일원화
# ---------------------------------------------------------------------------
class DataHandler:
    def __init__(self, tokenizer=None, stopwords=None):
        self.tokenizer = tokenizer
        self.raw_train_data = None
        self.raw_test_data = None
        self.stopwords = stopwords if stopwords is not None else []

        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.word_index = None

    def load_data(self, train_data_path: str, test_data_path: str):
        if os.path.exists(train_data_path) and os.path.exists(test_data_path):
            self.raw_train_data = pd.read_table(train_data_path)
            self.raw_test_data = pd.read_table(test_data_path)
        else:
            raise FileNotFoundError(f"⚠️ 데이터 파일 경로를 다시 확인해주세요.\nTrain: {train_data_path}\nTest: {test_data_path}")

    def clean_and_drop(self):
        self.raw_train_data.drop_duplicates(subset=['document'], inplace=True)
        self.raw_test_data.drop_duplicates(subset=['document'], inplace=True)
        self.raw_train_data.dropna(how='any', inplace=True)        
        self.raw_test_data.dropna(how='any', inplace=True)

    def process_konlpy(self, num_words=10000):
        """Mecab, Kkma, Okt 인스턴스를 가변 수용하는 통합 형태소 분석 전처리"""
        self.clean_and_drop()
        if self.tokenizer is None:
            raise ValueError("KoNLPy 형태소 훈련 분석을 위해 tokenizer 인스턴스 주입이 필수적입니다.")

        X_train_tokens = []
        for sentence in self.raw_train_data['document']:
            temp_X = self.tokenizer.morphs(str(sentence))
            temp_X = [word for word in temp_X if word not in self.stopwords]
            X_train_tokens.append(temp_X)

        X_test_tokens = []
        for sentence in self.raw_test_data['document']:
            temp_X = self.tokenizer.morphs(str(sentence))
            temp_X = [word for word in temp_X if word not in self.stopwords]
            X_test_tokens.append(temp_X)

        # 사전 및 정수 인덱스 빌드
        words = np.concatenate(X_train_tokens).tolist()
        counter = Counter(words).most_common(num_words - 4)
        
        vocab = ['<PAD>', '<BOS>', '<UNK>', '<UNUSED>'] + [key for key, _ in counter]
        word_to_index = {word: index for index, word in enumerate(vocab)}

        def wordlist_to_indexlist(wordlist):
            return [word_to_index[word] if word in word_to_index else word_to_index['<UNK>'] for word in wordlist]

        self.X_train = list(map(wordlist_to_indexlist, X_train_tokens))
        self.X_test = list(map(wordlist_to_indexlist, X_test_tokens))
        
        self.X_train = [torch.tensor(seq) if len(seq) > 0 else torch.tensor([0]) for seq in self.X_train]
        self.X_test = [torch.tensor(seq) if len(seq) > 0 else torch.tensor([0]) for seq in self.X_test]

        self.y_train = np.array(list(self.raw_train_data['label']))
        self.y_test = np.array(list(self.raw_test_data['label']))
        self.word_index = {index: word for word, index in word_to_index.items()}

    def process_sentencepiece(self, sp_model, vocab_dir):
        self.clean_and_drop()
        
        # document와 label을 확실하게 리스트로 동시 확보하여 순서 고정
        documents = self.raw_train_data['document'].tolist()
        labels = self.raw_train_data['label'].tolist()
        
        X_train_tensor, word_index, index_word = sp_tokenize(sp_model, documents, vocab_dir=vocab_dir)
        
        self.X_train = X_train_tensor
        self.y_train = np.array(labels)  # 동기화된 레이블 주입
        
        # Test 데이터도 동일하게 처리
        teost_documents = self.raw_test_data['document'].tolist()
        test_labels = self.raw_test_data['label'].tolist()
        
        X_test_tensor, _, _ = sp_tokenize(sp_model, test_documents, vocab_dir=vocab_dir)
        self.X_test = X_test_tensor
        self.y_test = np.array(test_labels)
        
        self.word_index = index_wrd

    def get_preprocessed_train_test_data_and_word_to_index(self):
        return self.X_train, self.y_train, self.X_test, self.y_test, self.word_index


# ---------------------------------------------------------------------------
# LSTM 공통 모델 아키텍처
# ---------------------------------------------------------------------------
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, num_layers=1)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        return self.sigmoid(self.fc(hidden[-1])).squeeze(-1)


# ---------------------------------------------------------------------------
# TrainHandler: 실험 모델 트래킹 인프라
# ---------------------------------------------------------------------------
class TrainHandler:
    def __init__(self, X_train, y_train, X_test, y_test, word_index):
        self.raw_X_train = X_train
        self.raw_y_train = y_train
        self.raw_X_test = X_test
        self.raw_y_test = y_test
        self.word_index = word_index
        self.vocab_size = max(word_index.keys()) + 1
        
        self.X_train_padded = None
        self.X_test_padded = None
        self.train_loader = None
        self.val_loader = None
        self.test_loader = None
        
        self.models = {}
        self.histories = {}

    def prepare_tensors(self, max_len=50):
        if isinstance(self.raw_X_train, torch.Tensor):
            if max_len is not None and max_len < self.raw_X_train.shape[1]:
                self.X_train_padded = self.raw_X_train[:, :max_len]
                self.X_test_padded = self.raw_X_test[:, :max_len]
            else:
                self.X_train_padded = self.raw_X_train
                self.X_test_padded = self.raw_X_test
        else:
            X_train_truncated = [seq[:max_len] if len(seq) > 0 else torch.tensor([0]) for seq in self.raw_X_train]
            X_test_truncated = [seq[:max_len] if len(seq) > 0 else torch.tensor([0]) for seq in self.raw_X_test]
            
            self.X_train_padded = pad_sequence(X_train_truncated, batch_first=True, padding_value=0)
            self.X_test_padded = pad_sequence(X_test_truncated, batch_first=True, padding_value=0)

    def make_validation_set(self, val_ratio=0.2, batch_size=256):
        full_dataset = TensorDataset(self.X_train_padded.long(), torch.tensor(self.raw_y_train, dtype=torch.float32))
        test_dataset = TensorDataset(self.X_test_padded.long(), torch.tensor(self.raw_y_test, dtype=torch.float32))

        val_size = int(len(full_dataset) * val_ratio)
        train_size = len(full_dataset) - val_size

        train_dataset, val_dataset = random_split(
            full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
        )

        self.train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        self.val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        self.test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    def config_model(self, model_key, embedding_dim=100, hidden_dim=128):
        self.models[model_key] = LSTMModel(vocab_size=self.vocab_size, embedding_dim=embedding_dim, hidden_dim=hidden_dim).to(device)

    def train_model(self, model_key, epochs=5, lr=0.001, patience=2):
        model = self.models[model_key]
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None
        
        for epoch in range(epochs):
            # ==========================================
            # 1. 훈련 루프 (TRAIN LOOP)
            # ==========================================
            model.train()
            tr_loss = 0.0
            for inputs, labels in self.train_loader:
                inputs = inputs.to(device)
                labels = labels.to(device).float().view(-1)  # 👈 레이블 1D 벡터화 및 float32 보장
                
                optimizer.zero_grad()
                
                outputs = model(inputs).view(-1)  # 👈 모델 출력 1D 벡터화 ([Batch] 크기로 통일)
                loss = criterion(outputs, labels)
                
                loss.backward()
                optimizer.step()
                tr_loss += loss.item() * inputs.size(0)

            # ==========================================
            # 2. 검증 루프 (VALIDATION LOOP)
            # ==========================================
            model.eval()
            val_loss = 0.0
            val_correct = 0
            with torch.no_grad():
                for inputs, labels in self.val_loader:
                    inputs = inputs.to(device)
                    labels = labels.to(device).float().view(-1)
                    
                    outputs = model(inputs).view(-1)
                    val_loss += criterion(outputs, labels).item() * inputs.size(0)
                    
                    # 정확도(Accuracy) 계산을 위한 임계값 처리
                    preds = (outputs >= 0.5).float()
                    val_correct += (preds == labels).sum().item()
            
            epoch_tr_loss = tr_loss / len(self.train_loader.dataset)
            epoch_val_loss = val_loss / len(self.val_loader.dataset)
            epoch_val_acc = val_correct / len(self.val_loader.dataset)

            # 조기 종료(Early Stopping) 체크 및 최적 가중치 저장
            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break

        # 최적의 상태 복원
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        
        # ==========================================
        # 3. 최종 평가 루프 (TEST LOOP)
        # ==========================================
        model.eval()
        test_correct = 0
        with torch.no_grad():
            for inputs, labels in self.test_loader:
                inputs = inputs.to(device)
                labels = labels.to(device).float().view(-1)
                
                outputs = model(inputs).view(-1)
                preds = (outputs >= 0.5).float()
                test_correct += (preds == labels).sum().item()
        
        test_acc = test_correct / len(self.test_loader.dataset)
        print(f"📊 [{model_key}] Test Accuracy: {test_acc:.4f}")
        return test_acc

In [ ]:
if __name__ == "__main__":
    # 데이터셋 로드 정밀 절대 경로 지정
    DATA_DIR = os.path.join(os.getcwd(), '..', '..', 'work', 'sentiment_classification', 'data')
    TRAIN_PATH = os.path.join(DATA_DIR, 'ratings_train.txt')
    TEST_PATH = os.path.join(DATA_DIR, 'ratings_test.txt')

    # # -------------------------------------------------------------
    # # [안정성 보완] 쾌적한 릴레이 실험을 위한 샘플링 다운사이징 (선택)
    # # -------------------------------------------------------------
    # # 원본 파일이 너무 크고 Kkma가 느리므로 빠른 경향성 파악을 위해 3만 건만 샘플링
    # df_train_tmp = pd.read_table(TRAIN_PATH).dropna().sample(n=30000, random_state=42)
    # df_test_tmp = pd.read_table(TEST_PATH).dropna().sample(n=10000, random_state=42)
    
    # # 임시 파일로 저장하여 하위 파이프라인이 이 샘플을 바라보게 변경
    # TRAIN_PATH = os.path.join(DATA_DIR, 'ratings_train_sample.txt')
    # TEST_PATH = os.path.join(DATA_DIR, 'ratings_test_sample.txt')
    
    # df_train_tmp.to_csv(TRAIN_PATH, sep='\t', index=False)
    # df_test_tmp.to_csv(TEST_PATH, sep='\t', index=False)
    # # -------------------------------------------------------------

    # SPM 전용 저장 상대/절대 디렉토리 경로 지정
    SPM_DIR = os.path.join(os.getcwd(), '..', '..', 'work', 'data', 'ex06')
    os.makedirs(SPM_DIR, exist_ok=True)

    # 불용어 정의
    stopwords = ['의','가','이','은','들','는','과','도','를','으로','자','에','와','한','하다']
    
    # 최종 성능 스토리지
    performance_results = {}

    # =======================================================================
    # 파트 1. KoNLPy 형태소 분석기 3종 (Mecab, Kkma, Okt) 릴레이 벤치마크
    # =======================================================================
    konlpy_tokenizers = {
        'Mecab': Mecab(),
        # 'Kkma': Kkma(),
        # 'Okt': Okt()
    }

    for name, tokenizer_obj in konlpy_tokenizers.items():
        print(f"\n🚀 [실험 RUN] KoNLPy - {name} 파이프라인 가동 중...")
        try:
            handler = DataHandler(tokenizer=tokenizer_obj, stopwords=stopwords)
            handler.load_data(TRAIN_PATH, TEST_PATH)
            handler.process_konlpy(num_words=10000)
            
            X_tr, y_tr, X_te, y_te, w_idx = handler.get_preprocessed_train_test_data_and_word_to_index()
            
            trainer = TrainHandler(X_tr, y_tr, X_te, y_te, w_idx)
            trainer.prepare_tensors(max_len=50)
            trainer.make_validation_set(batch_size=256)
            trainer.config_model(name)
            
            acc = trainer.train_model(name, epochs=3)
            performance_results[name] = acc
        except Exception as e:
            print(f"❌ {name} 형태소 분석 중 에러 발생 (건너뜀): {e}")

    # =======================================================================
    # 파트 2. SentencePiece (SPM) 하이퍼파라미터 가변 교차 검증 실험
    # =======================================================================
    print("\n--- SentencePiece 학습 소스 추출 중 ---")
    df_src = pd.read_table(TRAIN_PATH).dropna()
    temp_corpus_path = os.path.join(SPM_DIR, "corpus_src.txt")
    with open(temp_corpus_path, "w", encoding="utf-8") as f:
        f.write("\n".join(df_src['document'].tolist()))

    # 실험 하이퍼파라미터 그리드 매트릭스 선언
    spm_experiments = [
        {"model_type": "unigram", "vocab_size": 8000},
        {"model_type": "unigram", "vocab_size": 16000},
        {"model_type": "bpe", "vocab_size": 8000},
        {"model_type": "bpe", "vocab_size": 16000},
    ]

    for exp in spm_experiments:
        m_type = exp["model_type"]
        v_size = exp["vocab_size"]
        exp_key = f"SPM_{m_type.upper()}_Vocab{v_size}"
        
        print(f"\n🚀 [실험 RUN] {exp_key} 토크나이저 빌드 및 훈련 가동...")
        
        # 1. 모델 개별 저장 생성 및 반환
        sp_model = train_spm_tokenizer(
            input_txt_path=temp_corpus_path, 
            vocab_size=v_size, 
            model_type=m_type, 
            output_dir=SPM_DIR
        )
        
        # 2. 전처리 핸들러 매핑
        spm_handler = DataHandler()
        spm_handler.load_data(TRAIN_PATH, TEST_PATH)
        spm_handler.process_sentencepiece(sp_model=sp_model, vocab_dir=SPM_DIR)
        
        X_tr, y_tr, X_te, y_te, w_idx = spm_handler.get_preprocessed_train_test_data_and_word_to_index()
        
        # 3. 데이터 로더 구성 및 가중치 수렴
        trainer = TrainHandler(X_tr, y_tr, X_te, y_te, w_idx)
        trainer.prepare_tensors(max_len=50)
        trainer.make_validation_set(batch_size=256)
        trainer.config_model(exp_key)
        
        acc = trainer.train_model(exp_key, epochs=3)
        performance_results[exp_key] = acc

    # =======================================================================
    # 파트 3. 최종 스코어보드 종합 리포트 출력
    # =======================================================================
    print("\n" + "="*60)
    print("🏆 토크나이저별 네이버 영화 감정 분석 최종 성능 스코어보드")
    print("="*60)
    for model_name, score in performance_results.items():
        print(f" ▹ {model_name:<25} -> Test Accuracy: {score:.4f}")
    print("="*60)


🚀 [실험 RUN] KoNLPy - Mecab 파이프라인 가동 중...
📊 [Mecab] Test Accuracy: 0.6560

--- SentencePiece 학습 소스 추출 중 ---

🚀 [실험 RUN] SPM_UNIGRAM_Vocab8000 토크나이저 빌드 및 훈련 가동...
📊 [SPM_UNIGRAM_Vocab8000] Test Accuracy: 0.6519

🚀 [실험 RUN] SPM_UNIGRAM_Vocab16000 토크나이저 빌드 및 훈련 가동...
📊 [SPM_UNIGRAM_Vocab16000] Test Accuracy: 0.6004

🚀 [실험 RUN] SPM_BPE_Vocab8000 토크나이저 빌드 및 훈련 가동...
📊 [SPM_BPE_Vocab8000] Test Accuracy: 0.5815

🚀 [실험 RUN] SPM_BPE_Vocab16000 토크나이저 빌드 및 훈련 가동...
📊 [SPM_BPE_Vocab16000] Test Accuracy: 0.6053

🏆 토크나이저별 네이버 영화 감정 분석 최종 성능 스코어보드
 ▹ Mecab                     -> Test Accuracy: 0.6560
 ▹ SPM_UNIGRAM_Vocab8000     -> Test Accuracy: 0.6519
 ▹ SPM_UNIGRAM_Vocab16000    -> Test Accuracy: 0.6004
 ▹ SPM_BPE_Vocab8000         -> Test Accuracy: 0.5815
 ▹ SPM_BPE_Vocab16000        -> Test Accuracy: 0.6053


In [30]:
if __name__ == "__main__":
    # 데이터셋 로드 정밀 절대 경로 지정
    DATA_DIR = os.path.join(os.getcwd(), '..', '..', 'work', 'sentiment_classification', 'data')
    TRAIN_PATH = os.path.join(DATA_DIR, 'ratings_train.txt')
    TEST_PATH = os.path.join(DATA_DIR, 'ratings_test.txt')

    # # -------------------------------------------------------------
    # # [안정성 보완] 쾌적한 릴레이 실험을 위한 샘플링 다운사이징 (선택)
    # # -------------------------------------------------------------
    # # 원본 파일이 너무 크고 Kkma가 느리므로 빠른 경향성 파악을 위해 3만 건만 샘플링
    # df_train_tmp = pd.read_table(TRAIN_PATH).dropna().sample(n=30000, random_state=42)
    # df_test_tmp = pd.read_table(TEST_PATH).dropna().sample(n=10000, random_state=42)
    
    # # 임시 파일로 저장하여 하위 파이프라인이 이 샘플을 바라보게 변경
    # TRAIN_PATH = os.path.join(DATA_DIR, 'ratings_train_sample.txt')
    # TEST_PATH = os.path.join(DATA_DIR, 'ratings_test_sample.txt')
    
    # df_train_tmp.to_csv(TRAIN_PATH, sep='\t', index=False)
    # df_test_tmp.to_csv(TEST_PATH, sep='\t', index=False)
    # # -------------------------------------------------------------

    # SPM 전용 저장 상대/절대 디렉토리 경로 지정
    SPM_DIR = os.path.join(os.getcwd(), '..', '..', 'work', 'data', 'ex06')
    os.makedirs(SPM_DIR, exist_ok=True)

    # 불용어 정의
    stopwords = ['의','가','이','은','들','는','과','도','를','으로','자','에','와','한','하다']
    
    # 최종 성능 스토리지
    performance_results = {}

    # =======================================================================
    # 파트 1. KoNLPy 형태소 분석기 3종 (Mecab, Kkma, Okt) 릴레이 벤치마크
    # =======================================================================
    konlpy_tokenizers = {
        'Mecab': Mecab(),
        # 'Kkma': Kkma(),
        # 'Okt': Okt()
    }

    for name, tokenizer_obj in konlpy_tokenizers.items():
        print(f"\n🚀 [실험 RUN] KoNLPy - {name} 파이프라인 가동 중...")
        try:
            handler = DataHandler(tokenizer=tokenizer_obj, stopwords=stopwords)
            handler.load_data(TRAIN_PATH, TEST_PATH)
            handler.process_konlpy(num_words=10000)
            
            X_tr, y_tr, X_te, y_te, w_idx = handler.get_preprocessed_train_test_data_and_word_to_index()
            
            trainer = TrainHandler(X_tr, y_tr, X_te, y_te, w_idx)
            trainer.prepare_tensors(max_len=50)
            trainer.make_validation_set(batch_size=256)
            trainer.config_model(name)
            
            acc = trainer.train_model(name, epochs=10)
            performance_results[name] = acc
        except Exception as e:
            print(f"❌ {name} 형태소 분석 중 에러 발생 (건너뜀): {e}")

    # =======================================================================
    # 파트 2. SentencePiece (SPM) 하이퍼파라미터 가변 교차 검증 실험
    # =======================================================================
    print("\n--- SentencePiece 학습 소스 추출 중 ---")
    df_src = pd.read_table(TRAIN_PATH).dropna()
    temp_corpus_path = os.path.join(SPM_DIR, "corpus_src.txt")
    with open(temp_corpus_path, "w", encoding="utf-8") as f:
        f.write("\n".join(df_src['document'].tolist()))

    # 실험 하이퍼파라미터 그리드 매트릭스 선언
    spm_experiments = [
        {"model_type": "unigram", "vocab_size": 8000},
        # {"model_type": "unigram", "vocab_size": 16000},
        {"model_type": "bpe", "vocab_size": 8000},
        # {"model_type": "bpe", "vocab_size": 16000},
    ]

    for exp in spm_experiments:
        m_type = exp["model_type"]
        v_size = exp["vocab_size"]
        exp_key = f"SPM_{m_type.upper()}_Vocab{v_size}"
        
        print(f"\n🚀 [실험 RUN] {exp_key} 토크나이저 빌드 및 훈련 가동...")
        
        # 1. 모델 개별 저장 생성 및 반환
        sp_model = train_spm_tokenizer(
            input_txt_path=temp_corpus_path, 
            vocab_size=v_size, 
            model_type=m_type, 
            output_dir=SPM_DIR
        )
        
        # 2. 전처리 핸들러 매핑
        spm_handler = DataHandler()
        spm_handler.load_data(TRAIN_PATH, TEST_PATH)
        spm_handler.process_sentencepiece(sp_model=sp_model, vocab_dir=SPM_DIR)
        
        X_tr, y_tr, X_te, y_te, w_idx = spm_handler.get_preprocessed_train_test_data_and_word_to_index()
        
        # 3. 데이터 로더 구성 및 가중치 수렴
        trainer = TrainHandler(X_tr, y_tr, X_te, y_te, w_idx)
        trainer.prepare_tensors(max_len=50)
        trainer.make_validation_set(batch_size=256)
        trainer.config_model(exp_key)
        
        acc = trainer.train_model(exp_key, epochs=10)
        performance_results[exp_key] = acc

    # =======================================================================
    # 파트 3. 최종 스코어보드 종합 리포트 출력
    # =======================================================================
    print("\n" + "="*60)
    print("🏆 토크나이저별 네이버 영화 감정 분석 최종 성능 스코어보드")
    print("="*60)
    for model_name, score in performance_results.items():
        print(f" ▹ {model_name:<25} -> Test Accuracy: {score:.4f}")
    print("="*60)


🚀 [실험 RUN] KoNLPy - Mecab 파이프라인 가동 중...
📊 [Mecab] Test Accuracy: 0.6121

--- SentencePiece 학습 소스 추출 중 ---

🚀 [실험 RUN] SPM_UNIGRAM_Vocab8000 토크나이저 빌드 및 훈련 가동...
📊 [SPM_UNIGRAM_Vocab8000] Test Accuracy: 0.6697

🚀 [실험 RUN] SPM_BPE_Vocab8000 토크나이저 빌드 및 훈련 가동...
📊 [SPM_BPE_Vocab8000] Test Accuracy: 0.6130

🏆 토크나이저별 네이버 영화 감정 분석 최종 성능 스코어보드
 ▹ Mecab                     -> Test Accuracy: 0.6121
 ▹ SPM_UNIGRAM_Vocab8000     -> Test Accuracy: 0.6697
 ▹ SPM_BPE_Vocab8000         -> Test Accuracy: 0.6130


In [ ]:
class Tokenizer:
    def __init__(self, filters=''):
        self.word_index = {}
        self.index_word = {}
        self.filters = filters

    def fit_on_texts(self, corpus):
        for sentence in corpus:
            tokens = sentence.split() if isinstance(sentence, str) else sentence
            for token in tokens:
                if token not in self.word_index:
                    self.word_index[token] = len(self.word_index) + 1
        self.index_word = {idx: word for word, idx in self.word_index.items()}

    def texts_to_sequences(self, corpus):
        sequences = []
        for sentence in corpus:
            tokens = sentence.split() if isinstance(sentence, str) else sentence
            seq = [self.word_index.get(token, 0) for token in tokens]
            sequences.append(torch.tensor(seq, dtype=torch.long))
        return sequences

    def sequences_to_texts(self, sequences):
        texts = []
        for seq in sequences:
            if isinstance(seq, torch.Tensor):
                seq = seq.tolist()
            tokens = [self.index_word.get(idx, "") for idx in seq if idx != 0]
            texts.append(tokens)
        return texts

def tokenize(corpus):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(corpus)
    sequences = tokenizer.texts_to_sequences(corpus)
    tensor = pad_sequence(sequences, batch_first=True, padding_value=0)
    return tensor, tokenizer

# def sp_tokenize(s, corpus):

#     tensor = []

#     for sen in corpus:
#         tensor.append(s.EncodeAsIds(sen))

#     with open("./korean_spm.vocab", 'r') as f:
#         vocab = f.readlines()

#     word_index = {}
#     index_word = {}

#     for idx, line in enumerate(vocab):
#         word = line.split("\t")[0]

#         word_index.update({word:idx})
#         index_word.update({idx:word})

#     tensor = pad_sequence(tensor, batch_first=True, padding_value=0)

#     return tensor, word_index, index_word

In [6]:
base_path = os.path.join(os.getcwd(), '..', '..', 'work', 'data', 'ex06')
path_to_file = os.path.join(base_path, "korean-english-park.train.ko")

with open(path_to_file, "r") as f:
    raw = f.read().splitlines()

print("Data Size:", len(raw))

print("Example:")
for sen in raw[0:100][::20]: print(">>", sen)

min_len = 999
max_len = 0
sum_len = 0

cleaned_corpus = list(set(raw))  # set를 사용해서 중복을 제거합니다.
print("Data Size:", len(cleaned_corpus))

for sen in cleaned_corpus:
    length = len(sen)
    if min_len > length: min_len = length
    if max_len < length: max_len = length
    sum_len += length

print("문장의 최단 길이:", min_len)
print("문장의 최장 길이:", max_len)
print("문장의 평균 길이:", sum_len // len(cleaned_corpus))

sentence_length = np.zeros((max_len), dtype=int)

for sen in cleaned_corpus:   # 중복이 제거된 코퍼스 기준
    sentence_length[len(sen)-1] += 1

Data Size: 94123
Example:
>> 개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"
>> 북한의 핵무기 계획을 포기하도록 하려는 압력이 거세지고 있는 가운데, 일본과 북한의 외교관들이 외교 관계를 정상화하려는 회담을 재개했다.
>> "경호 로보트가 침입자나 화재를 탐지하기 위해서 개인적으로, 그리고 전문적으로 사용되고 있습니다."
>> 수자원부 당국은 논란이 되고 있고, 막대한 비용이 드는 이 사업에 대해 내년에 건설을 시작할 계획이다.
>> 또한 근력 운동은 활발하게 걷는 것이나 최소한 20분 동안 뛰는 것과 같은 유산소 활동에서 얻는 운동 효과를 심장과 폐에 주지 않기 때문에, 연구학자들은 근력 운동이 심장에 큰 영향을 미치는지 여부에 대해 논쟁을 해왔다.
Data Size: 77591
문장의 최단 길이: 1
문장의 최장 길이: 377
문장의 평균 길이: 64


In [7]:
filtered_corpus = cleaned_corpus

In [8]:
temp_file = 'korean-english-park.train.ko.temp'

vocab_size = 8000

with open(temp_file, 'w') as f:
    for row in filtered_corpus:   # 이전에 나왔던 정제했던 corpus를 활용해서 진행해야 합니다.
        f.write(str(row) + '\n')

spm.SentencePieceTrainer.Train(
    '--input={} --model_prefix=korean_spm --vocab_size={}'.format(temp_file, vocab_size)    
)
#위 Train에서  --model_type = unigram이 디폴트 적용되어 있습니다. --model_type = bpe로 옵션을 주어 변경할 수 있습니다.

!ls -l korean_spm*

-rw-r--r-- 1 ysoh1113 ysoh1113 379738 Jun 15 14:22 korean_spm.model
-rw-r--r-- 1 ysoh1113 ysoh1113 146802 Jun 15 14:22 korean_spm.vocab


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=korean-english-park.train.ko.temp --model_prefix=korean_spm --vocab_size=8000
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: korean-english-park.train.ko.temp
  input_format: 
  model_prefix: korean_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  b

In [9]:
s = spm.SentencePieceProcessor()
s.Load('korean_spm.model')

# SentencePiece를 활용한 sentence -> encoding
tokensIDs = s.EncodeAsIds('아버지가방에들어가신다.')
print(tokensIDs)

# SentencePiece를 활용한 sentence -> encoded pieces
print(s.SampleEncodeAsPieces('아버지가방에들어가신다.',-1, 0.1))

# SentencePiece를 활용한 encoding -> sentence 복원
print(s.DecodeIds(tokensIDs))

[1261, 11, 301, 7, 3557, 11, 290, 33, 4]
['▁아버지', '가', '방', '에', '들어', '가', '신', '다', '.']
아버지가방에들어가신다.
